In [1]:
!pip install torch transformers torchaudio librosa


/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)


In [1]:
import torch
import librosa
import numpy as np
from transformers import WhisperProcessor, WhisperModel

# Definir dispositivo (GPU se disponível)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Carregar modelo Whisper
# model_name = "openai/whisper-large-v3"
# model_name = "openai/whisper-medium"
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name).to(device)

def extract_whisper_features(audio_path, sample_rate=16000):
    # Carregar áudio e converter para tensor
    waveform, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    inputs = processor(waveform, return_tensors="pt", sampling_rate=sample_rate).input_features.to(device)

    # Extrair embeddings
    with torch.no_grad():
        embeddings = model.encoder(inputs).last_hidden_state  # (batch, seq_len, hidden_dim)

    return embeddings.squeeze(0).cpu().numpy()  # (seq_len, hidden_dim)

# Teste
audio_path = "teste2.wav"
whisper_features = extract_whisper_features(audio_path)
print("Formato dos embeddings Whisper:", whisper_features.shape)


/home/alex/miniconda3/envs/AudioTransformers/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Formato dos embeddings Whisper: (1500, 768)


In [2]:
whisper_features.shape

(1500, 768)

In [10]:
import torch

torch.cuda.empty_cache()  # Libera cache da GPU
torch.cuda.ipc_collect()  # Coleta memória não utilizada

print("Memória da GPU liberada!")


Memória da GPU liberada!


In [11]:
import torch

torch.cuda.empty_cache()  # Libera cache da GPU
torch.cuda.memory_reserved()  # Mostra memória reservada
torch.cuda.memory_allocated()  # Mostra memória alocada


976234496

In [12]:
import gc
import torch

gc.collect()  # Limpar objetos não referenciados na RAM
torch.cuda.empty_cache()  # Limpar cache da GPU

In [13]:
import torch
import torchaudio
from transformers import (
    Wav2Vec2FeatureExtractor, Wav2Vec2Model, WavLMModel, HubertModel, Data2VecAudioModel
)

# Função para carregar áudio e converter para tensor de 16kHz
def load_audio(file_path, target_sample_rate=16000):
    waveform, sample_rate = torchaudio.load(file_path)

    if sample_rate != target_sample_rate:
        transform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = transform(waveform)

    return waveform.squeeze(0), target_sample_rate

# Extração de características com WavLM
def extract_features_wavlm(file_path):
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")
    model = WavLMModel.from_pretrained("microsoft/wavlm-base")

    waveform, sample_rate = load_audio(file_path)
    inputs = feature_extractor(waveform, sampling_rate=sample_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.squeeze(0)  # Retorna as features extraídas

# Extração de características com wav2vec 2.0
def extract_features_wav2vec(file_path):
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
    model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

    waveform, sample_rate = load_audio(file_path)
    inputs = feature_extractor(waveform, sampling_rate=sample_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.squeeze(0)  # Retorna as features extraídas

# Extração de características com HuBERT
def extract_features_hubert(file_path):
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
    model = HubertModel.from_pretrained("facebook/hubert-base-ls960")

    waveform, sample_rate = load_audio(file_path)
    inputs = feature_extractor(waveform, sampling_rate=sample_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.squeeze(0)  # Retorna as features extraídas

# Extração de características com Data2Vec-Audio
def extract_features_data2vec(file_path):
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/data2vec-audio-base")
    model = Data2VecAudioModel.from_pretrained("facebook/data2vec-audio-base")

    waveform, sample_rate = load_audio(file_path)
    inputs = feature_extractor(waveform, sampling_rate=sample_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.squeeze(0)  # Retorna as features extraídas

# Caminho do arquivo de áudio
audio_file = "teste2.wav"

# Extração com WavLM
wavlm_features = extract_features_wavlm(audio_file)
print("Shape das features do WavLM:", wavlm_features.shape)

# Extração com wav2vec 2.0
wav2vec_features = extract_features_wav2vec(audio_file)
print("Shape das features do wav2vec:", wav2vec_features.shape)

# Extração com HuBERT
hubert_features = extract_features_hubert(audio_file)
print("Shape das features do HuBERT:", hubert_features.shape)

# Extração com Data2Vec-Audio
data2vec_features = extract_features_data2vec(audio_file)
print("Shape das features do Data2Vec-Audio:", data2vec_features.shape)


/home/alex/miniconda3/envs/AudioTransformers/lib/python3.9/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Shape das features do WavLM: torch.Size([221, 768])


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Shape das features do wav2vec: torch.Size([221, 768])
Shape das features do HuBERT: torch.Size([221, 768])
Shape das features do Data2Vec-Audio: torch.Size([221, 768])


In [8]:
wavlm_features

tensor([[-0.3196,  0.0719, -0.3837,  ...,  0.5026,  0.5484, -0.1396],
        [-0.2550,  0.0922, -0.3205,  ...,  0.4657,  0.5129, -0.1325],
        [-0.1838,  0.0865, -0.3402,  ...,  0.4827,  0.4742, -0.1199],
        ...,
        [-0.5738,  0.0781, -0.4956,  ..., -0.6192,  0.1576, -0.3082],
        [-0.3487,  0.0580, -0.3026,  ..., -0.0033,  0.4154, -0.2648],
        [-0.3623,  0.0492, -0.2526,  ...,  0.1310,  0.4320, -0.2924]])